# 🏎️ JetBot Mario Kart — Racing Controller

**Author:** Racing subsystem  
**Collaborator:** Jake (Vision / Lane Detection)  
**Track:** Oval — clockwise direction

This notebook implements the autonomous racing controller for the JetBot Mario Kart project.
It consumes lane detection output from Jake's vision module and drives the robot around the track.

### Architecture
```
Jake's Vision  →  LaneInput  →  OvalRacingController (PID + phase)  →  JetBot Motors
                                            ↑
                                    RaceSession (loop + state + laps)
```

### Oval-specific design decisions
- **Clockwise bias**: The bot always turns right, so a small `CLOCKWISE_BIAS` is added
  to the steering each frame. This pre-loads the PID instead of waiting for integral wind-up.
- **Straight vs corner phase**: Offset magnitude tells us where we are on the oval.
  Small offset → we're on a straight → use `STRAIGHT_SPEED`.
  Large offset → we're in a corner → use `CORNER_SPEED` (slower).
- **Asymmetric integral clamp**: Left-turn integral wind-up is clamped tightly since
  on a clockwise oval the bot should almost never need sustained left correction.
- **Lap timer**: Laps are counted by time (simple, reliable) with an optional
  start/finish line hook for Jake's vision if you add that later.

### Interface contract with Jake
Jake's vision module should populate a `LaneInput` object each frame:
- `offset` : float in **[-1.0, +1.0]** — how far the lane center is from image center
  - `-1.0` = lane is far left  →  robot needs to steer **left**
  - ` 0.0` = lane is centered  →  robot goes **straight**
  - `+1.0` = lane is far right →  robot needs to steer **right**
- `confidence` : float in **[0.0, 1.0]** — how certain the detection is
- `lane_detected` : bool — False triggers clockwise recovery spin

## Cell 1 — Imports & JetBot setup

In [ ]:
import time
import threading
from dataclasses import dataclass
from enum import Enum, auto
from typing import Optional

# JetBot motor library
from jetbot import Robot

# Notebook display utilities
import ipywidgets as widgets
from IPython.display import display

print("✅ Imports OK")

## Cell 2 — Shared data structures (interface with Jake)

In [ ]:
@dataclass
class LaneInput:
    """
    Data produced by Jake's vision module each frame.

    Jake: populate these fields in your detection loop, then call
          race_session.update_lane(lane_input) once per frame.
    """
    offset: float = 0.0          # -1.0 (left) .. 0.0 (center) .. +1.0 (right)
    confidence: float = 1.0      #  0.0 (uncertain) .. 1.0 (very confident)
    lane_detected: bool = True   # False = no lane visible this frame


@dataclass
class MotorCommand:
    """Output of the controller — motor speeds in [-1.0, +1.0]."""
    left: float = 0.0
    right: float = 0.0


class TrackPhase(Enum):
    """Which part of the oval the bot currently estimates it is on."""
    STRAIGHT = auto()   # Small lane offset -> long straight
    CORNER   = auto()   # Large lane offset -> banked corner


class RaceState(Enum):
    WAITING   = auto()   # Before race starts
    RACING    = auto()   # Normal lap driving
    SEARCHING = auto()   # Lane lost — performing clockwise recovery spin
    STOPPED   = auto()   # Race ended or emergency stop


print("✅ Data structures defined")

## Cell 3 — PID Controller (asymmetric integral clamp for clockwise oval)

In [ ]:
class PIDController:
    """
    PID controller tuned for a clockwise oval track.

    The integral clamp is asymmetric:
      - Right turn integral can wind up to +integral_max_right (bot turns right freely)
      - Left turn integral is clamped tightly to -integral_max_left
        because on a clockwise oval the bot should almost never need sustained left
        correction. Limiting the integral prevents the bot from over-correcting inward.

    Args:
        Kp: Proportional gain  — how hard to correct present error
        Ki: Integral gain      — corrects steady drift over time
        Kd: Derivative gain    — dampens oscillation / sudden changes
        integral_max_right: Max positive integral accumulation (right turns)
        integral_max_left:  Max negative integral accumulation (left turns, keep small)
    """

    def __init__(
        self,
        Kp: float,
        Ki: float,
        Kd: float,
        integral_max_right: float = 1.0,
        integral_max_left:  float = 0.3,
    ):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.integral_max_right = integral_max_right
        self.integral_max_left  = integral_max_left

        self._integral   = 0.0
        self._prev_error = 0.0
        self._prev_time  = None

    def reset(self):
        """Call at race start or after a lane-lost recovery."""
        self._integral   = 0.0
        self._prev_error = 0.0
        self._prev_time  = None

    def compute(self, error: float) -> float:
        """
        Compute steering correction given the current lane error.

        Args:
            error: lane offset [-1.0, +1.0], positive = steer right

        Returns:
            steering: positive = steer right, negative = steer left
        """
        now = time.monotonic()

        if self._prev_time is None:
            dt = 0.033
        else:
            dt = max(now - self._prev_time, 1e-6)

        self._integral += error * dt

        # Asymmetric clamp: generous right wind-up, tight left wind-up
        self._integral = max(
            -self.integral_max_left,
            min(self.integral_max_right, self._integral)
        )

        derivative = (error - self._prev_error) / dt

        self._prev_error = error
        self._prev_time  = now

        return self.Kp * error + self.Ki * self._integral + self.Kd * derivative


print("✅ PID controller defined")

## Cell 4 — Oval Racing Controller

In [ ]:
class OvalRacingController:
    """
    Racing controller optimised for a clockwise oval track.

    Key features vs a generic lane-follower:
      1. Clockwise steering bias: a small fixed right-turn offset added every frame
         so the PID starts from a better initial state going into corners.
      2. Phase-aware speed: detects STRAIGHT vs CORNER from offset magnitude,
         then applies the appropriate target speed.
      3. Smooth speed transitions: speed changes are rate-limited so the bot does
         not lurch when entering or exiting a corner.
    """

    def __init__(
        self,
        straight_speed:    float = 0.45,
        corner_speed:      float = 0.30,
        max_speed:         float = 0.60,
        steering_gain:     float = 0.60,
        clockwise_bias:    float = 0.08,   # constant right-turn nudge
        corner_threshold:  float = 0.25,   # |offset| above this = corner mode
        min_confidence:    float = 0.30,
        speed_slew_rate:   float = 0.05,   # max speed change per frame
        pid: Optional[PIDController] = None,
    ):
        self.straight_speed   = straight_speed
        self.corner_speed     = corner_speed
        self.max_speed        = max_speed
        self.steering_gain    = steering_gain
        self.clockwise_bias   = clockwise_bias
        self.corner_threshold = corner_threshold
        self.min_confidence   = min_confidence
        self.speed_slew_rate  = speed_slew_rate
        self.pid              = pid or PIDController(Kp=0.6, Ki=0.02, Kd=0.1)

        self._current_speed = straight_speed
        self.current_phase  = TrackPhase.STRAIGHT

    def compute(self, lane: LaneInput) -> MotorCommand:
        """Return motor speeds for this frame given Jake's lane detection."""

        # Low confidence or lane lost -> full stop
        if not lane.lane_detected or lane.confidence < self.min_confidence:
            return MotorCommand(left=0.0, right=0.0)

        # 1. Determine track phase from offset magnitude
        if abs(lane.offset) > self.corner_threshold:
            self.current_phase = TrackPhase.CORNER
            target_speed = self.corner_speed
        else:
            self.current_phase = TrackPhase.STRAIGHT
            target_speed = self.straight_speed

        # 2. Slew speed toward target (no lurching)
        delta = target_speed - self._current_speed
        delta = max(-self.speed_slew_rate, min(self.speed_slew_rate, delta))
        self._current_speed += delta

        # Scale by confidence
        speed = self._current_speed * lane.confidence

        # 3. PID steering + constant clockwise bias
        pid_correction = self.pid.compute(lane.offset)
        steering = pid_correction + self.clockwise_bias

        # 4. Differential steering
        #    positive steering -> turn right -> left motor faster, right motor slower
        left_speed  = speed + steering * self.steering_gain
        right_speed = speed - steering * self.steering_gain

        left_speed  = max(-self.max_speed, min(self.max_speed, left_speed))
        right_speed = max(-self.max_speed, min(self.max_speed, right_speed))

        return MotorCommand(left=left_speed, right=right_speed)

    def reset(self):
        self.pid.reset()
        self._current_speed = self.straight_speed
        self.current_phase  = TrackPhase.STRAIGHT


print("✅ OvalRacingController defined")

## Cell 5 — Tunable parameters

| Parameter | Typical range | Effect |
|---|---|---|
| `STRAIGHT_SPEED` | 0.35 – 0.55 | Speed on long straights |
| `CORNER_SPEED` | 0.25 – 0.40 | Speed in corners (should be < STRAIGHT_SPEED) |
| `CLOCKWISE_BIAS` | 0.04 – 0.15 | Right-turn nudge every frame. Too high = bot hugs outer wall |
| `CORNER_THRESHOLD` | 0.15 – 0.40 | How large the offset must be before corner mode activates |
| `KP` | 0.3 – 1.0 | Snappier turns; too high causes oscillation |
| `KI` | 0.0 – 0.05 | Corrects slow drift; keep low on an oval |
| `KD` | 0.0 – 0.3 | Dampens wiggle |
| `LAP_TIME_ESTIMATE` | measured seconds | Used for timer-based lap counting |

> **Clockwise bias tip:** Start at 0.0, run one lap, and note if the bot drifts to the
> outside of corners. Increase by 0.02 at a time until corners feel natural.

In [ ]:
# ── Tune these ────────────────────────────────────────────────
STRAIGHT_SPEED         = 0.42
CORNER_SPEED           = 0.28
MAX_SPEED              = 0.60
CLOCKWISE_BIAS         = 0.08    # right-turn nudge for clockwise oval
CORNER_THRESHOLD       = 0.25    # |offset| above this triggers CORNER phase
KP                     = 0.6
KI                     = 0.02
KD                     = 0.1
STEERING_GAIN          = 0.6
MIN_CONFIDENCE         = 0.3
SPEED_SLEW_RATE        = 0.05

# Recovery — spin clockwise (same direction as track)
RECOVERY_SPIN_SPEED    = 0.25
RECOVERY_SPIN_DURATION = 0.4    # seconds per spin pulse

# Lap counting
# Set to measured lap time in seconds for timer-based counting.
# Set to 0 to rely solely on Jake's start/finish line detection.
LAP_TIME_ESTIMATE      = 12.0
# ──────────────────────────────────────────────────────────────

print(f"Parameters set — Kp={KP}, Ki={KI}, Kd={KD}")
print(f"Speeds: straight={STRAIGHT_SPEED}, corner={CORNER_SPEED}, bias={CLOCKWISE_BIAS}")

## Cell 6 — Race Session (state machine + lap counter + dashboard)

In [ ]:
class RaceSession:
    """
    Manages the full race lifecycle on a clockwise oval.

    Lap counting (two options):
      A) Timer-based (default): counts a lap every LAP_TIME_ESTIMATE seconds.
         Calibrate LAP_TIME_ESTIMATE in Cell 5.
      B) Vision-based: Jake calls race_session.lap_complete() when he detects
         the start/finish line. Set LAP_TIME_ESTIMATE = 0 to disable timer.

    Usage:
        session = RaceSession(robot)
        session.start()
        # Jake calls session.update_lane(lane_input) in his loop
        session.stop()
    """

    def __init__(self, robot: Robot):
        self.robot      = robot
        self.controller = OvalRacingController(
            straight_speed   = STRAIGHT_SPEED,
            corner_speed     = CORNER_SPEED,
            max_speed        = MAX_SPEED,
            steering_gain    = STEERING_GAIN,
            clockwise_bias   = CLOCKWISE_BIAS,
            corner_threshold = CORNER_THRESHOLD,
            min_confidence   = MIN_CONFIDENCE,
            speed_slew_rate  = SPEED_SLEW_RATE,
            pid              = PIDController(
                Kp=KP, Ki=KI, Kd=KD,
                integral_max_right=1.0,
                integral_max_left=0.3,
            ),
        )

        self.state       = RaceState.WAITING
        self.lap_count   = 0
        self._lock       = threading.Lock()
        self._race_start = None
        self._last_lap_t = None
        self._lost_since = None

        # Dashboard widgets
        self._w_state  = widgets.Label(value="State: WAITING")
        self._w_laps   = widgets.Label(value="Laps: 0")
        self._w_phase  = widgets.Label(value="Phase: —")
        self._w_offset = widgets.FloatProgress(
            value=0.5, min=0.0, max=1.0,
            description="Lane offset",
            bar_style="info",
            layout=widgets.Layout(width="340px")
        )
        self._w_conf = widgets.FloatProgress(
            value=1.0, min=0.0, max=1.0,
            description="Confidence",
            bar_style="success",
            layout=widgets.Layout(width="340px")
        )
        self._w_left  = widgets.Label(value="L motor: 0.00")
        self._w_right = widgets.Label(value="R motor: 0.00")
        self._w_time  = widgets.Label(value="Race time: 0.0 s")
        self._btn_stop = widgets.Button(
            description="STOP",
            button_style="danger",
            layout=widgets.Layout(width="100px", height="40px")
        )
        self._btn_stop.on_click(lambda _: self.stop())

    # ── Public API ────────────────────────────────────────────

    def start(self):
        """Begin racing. Call once Jake's vision loop is producing frames."""
        self.controller.reset()
        self.lap_count   = 0
        self._lost_since = None
        self._race_start = time.monotonic()
        self._last_lap_t = self._race_start
        self.state       = RaceState.RACING
        self._display_dashboard()
        print("Race started! Clockwise oval mode.")

    def stop(self):
        """Emergency stop — cuts motors immediately."""
        self.state = RaceState.STOPPED
        self.robot.stop()
        self._w_state.value = "State: STOPPED"
        elapsed = time.monotonic() - self._race_start if self._race_start else 0
        print(f"Stopped after {elapsed:.1f} s, {self.lap_count} laps.")

    def update_lane(self, lane: LaneInput):
        """
        Jake calls this once per vision frame.

        Example:
            lane = LaneInput(offset=x_err, confidence=conf, lane_detected=True)
            race_session.update_lane(lane)
        """
        with self._lock:
            pass
        self._step(lane)

    def lap_complete(self):
        """
        Optional: Jake calls this when he detects the start/finish line.
        If using timer-based laps (LAP_TIME_ESTIMATE > 0) this is not required.
        """
        self._record_lap()

    # ── Internal logic ────────────────────────────────────────

    def _step(self, lane: LaneInput):
        if self.state == RaceState.STOPPED:
            return

        now = time.monotonic()

        # Timer-based lap counting
        if (
            self.state == RaceState.RACING
            and LAP_TIME_ESTIMATE > 0
            and self._last_lap_t is not None
            and (now - self._last_lap_t) >= LAP_TIME_ESTIMATE
        ):
            self._record_lap()

        if self.state == RaceState.RACING:
            if not lane.lane_detected or lane.confidence < MIN_CONFIDENCE:
                self._lost_since = now
                self.state = RaceState.SEARCHING
                self._recovery_spin()
                return

            cmd = self.controller.compute(lane)
            self._apply(cmd)
            self._update_dashboard(lane, cmd, now)

        elif self.state == RaceState.SEARCHING:
            if lane.lane_detected and lane.confidence >= MIN_CONFIDENCE:
                self.controller.reset()
                self.state = RaceState.RACING
                print("Lane re-acquired — resuming")
            else:
                elapsed = now - (self._lost_since or now)
                if elapsed > 3.0:
                    self.stop()
                    print("Lane lost for 3 s — stopping for safety")

    def _recovery_spin(self):
        """
        Spin clockwise (right) to search for the lane.
        On a clockwise oval, turning right is always the correct search direction.
        """
        print("Lane lost — spinning clockwise to search...")
        self._w_state.value = "State: SEARCHING"
        # Turn right: left motor forward, right motor backward
        self.robot.left_motor.value  =  RECOVERY_SPIN_SPEED
        self.robot.right_motor.value = -RECOVERY_SPIN_SPEED
        time.sleep(RECOVERY_SPIN_DURATION)
        self.robot.stop()

    def _record_lap(self):
        self.lap_count += 1
        self._last_lap_t = time.monotonic()
        self._w_laps.value = f"Laps: {self.lap_count}"
        print(f"Lap {self.lap_count} complete!")

    def _apply(self, cmd: MotorCommand):
        self.robot.left_motor.value  = cmd.left
        self.robot.right_motor.value = cmd.right

    def _display_dashboard(self):
        display(widgets.VBox([
            widgets.HBox([self._w_state, self._w_laps, self._w_phase, self._btn_stop]),
            self._w_offset,
            self._w_conf,
            widgets.HBox([self._w_left, self._w_right, self._w_time]),
        ]))

    def _update_dashboard(self, lane: LaneInput, cmd: MotorCommand, now: float):
        self._w_state.value  = f"State: {self.state.name}"
        self._w_phase.value  = f"Phase: {self.controller.current_phase.name}"
        self._w_offset.value = (lane.offset + 1.0) / 2.0
        self._w_conf.value   = lane.confidence
        self._w_conf.bar_style = "success" if lane.confidence > 0.6 else "warning"
        self._w_left.value   = f"L motor: {cmd.left:+.3f}"
        self._w_right.value  = f"R motor: {cmd.right:+.3f}"
        if self._race_start:
            self._w_time.value = f"Race time: {now - self._race_start:.1f} s"


print("✅ RaceSession defined")

## Cell 7 — Initialize robot & session

In [ ]:
robot = Robot()
race_session = RaceSession(robot)

print("✅ Robot and RaceSession ready")
print("   Call race_session.start() when Jake's vision loop is running")

## Cell 8 — Start the race

Run this cell **after** Jake's vision notebook is producing `LaneInput` values.  
The STOP button in the dashboard (or calling `race_session.stop()`) kills the motors instantly.

In [ ]:
race_session.start()

## Cell 9 — Jake's integration stub + simulation test

In [ ]:
# ── Integration stub for Jake ─────────────────────────────────
#
# In Jake's vision loop, after computing the lane offset:
#
#   lane = LaneInput(
#       offset        = x_error,              # -1.0 ... +1.0
#       confidence    = detection_confidence, # 0.0 ... 1.0
#       lane_detected = detected,             # bool
#   )
#   race_session.update_lane(lane)
#
# Optional: if Jake detects the start/finish line:
#   race_session.lap_complete()
#
# ─────────────────────────────────────────────────────────────

# Simulation test — runs without a real robot
# Simulates representative frames around a clockwise oval lap:
#   straight -> right corner -> straight -> right corner -> lap
print("Clockwise oval simulation test")
print()

sim_ctrl = OvalRacingController(
    straight_speed   = STRAIGHT_SPEED,
    corner_speed     = CORNER_SPEED,
    clockwise_bias   = CLOCKWISE_BIAS,
    corner_threshold = CORNER_THRESHOLD,
    pid              = PIDController(Kp=KP, Ki=KI, Kd=KD),
)

oval_frames = [
    ("Straight (centered)",     LaneInput(offset= 0.00, confidence=1.0,  lane_detected=True)),
    ("Straight (slight drift)", LaneInput(offset=-0.10, confidence=1.0,  lane_detected=True)),
    ("Entering right corner",   LaneInput(offset= 0.28, confidence=0.9,  lane_detected=True)),
    ("Deep in right corner",    LaneInput(offset= 0.55, confidence=0.85, lane_detected=True)),
    ("Exiting corner",          LaneInput(offset= 0.20, confidence=0.9,  lane_detected=True)),
    ("Back on straight",        LaneInput(offset= 0.05, confidence=1.0,  lane_detected=True)),
    ("Low confidence (shadow)", LaneInput(offset= 0.10, confidence=0.35, lane_detected=True)),
    ("Lane lost",               LaneInput(offset= 0.00, confidence=0.0,  lane_detected=False)),
]

print(f"{'Scenario':<30} {'Phase':<10} {'Offset':>7}  {'Conf':>5}  {'Left':>8}  {'Right':>8}")
print("-" * 76)
for label, lane in oval_frames:
    cmd   = sim_ctrl.compute(lane)
    phase = sim_ctrl.current_phase.name if lane.lane_detected else "—"
    print(f"{label:<30} {phase:<10} {lane.offset:>7.2f}  {lane.confidence:>5.2f}  {cmd.left:>+8.3f}  {cmd.right:>+8.3f}")

## Cell 10 — Emergency stop

Run this at **any time** to cut motors immediately.

In [ ]:
race_session.stop()

---
## Tuning guide for a clockwise oval

### Step 1 — Calibrate lap time
Push the robot around the track by hand and time it. Set `LAP_TIME_ESTIMATE` to that value.
Later, if Jake adds start/finish line detection, set `LAP_TIME_ESTIMATE = 0` to switch over.

### Step 2 — Set clockwise bias first
Start with `CLOCKWISE_BIAS = 0.0` and watch the first few corners. If the bot runs wide
(toward the outer wall), increase by 0.02 at a time. Typical good value: 0.06–0.10.

### Step 3 — Set corner threshold
Watch the Phase readout in the dashboard. It should flip to CORNER when the bot enters
the curves and STRAIGHT on the long runs. Adjust `CORNER_THRESHOLD` to match.

### Step 4 — Speed differential
`CORNER_SPEED` should be about 60–70% of `STRAIGHT_SPEED`. Widen the gap if the bot
slides wide in corners.

### Step 5 — Tune Kp (Ki=0, Kd=0 initially)
- Too low: drifts off track on straights
- Too high: oscillates left-right

### Step 6 — Add Kd to smooth corners
If the bot wiggles through corners, raise `KD` in steps of 0.05.

### Step 7 — Add Ki only for persistent drift
A small `KI = 0.01–0.03` is enough on an oval. The asymmetric integral clamp prevents
it from over-correcting inward.